# Notebook 3: Attention Weights
**LLM Fundamentals Demo Series — Agentic AI Bootcamp**

This notebook extracts and visualizes the **self-attention weights** from **DistilBERT** (66M params).

Topics covered:
1. How attention weights are computed (QKV review)
2. Extracting attention matrices from a transformer layer
3. Heatmap visualization of attention patterns
4. Comparing attention across layers
5. How attention resolves pronoun references (coreference)
6. Attention rollout — propagating attention across all layers

In [ ]:
!pip install transformers torch matplotlib seaborn --quiet

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()

print(f'Model : {MODEL_NAME}')
print(f'Layers: {model.config.num_hidden_layers}')
print(f'Heads : {model.config.num_attention_heads}')
print(f'Hidden: {model.config.hidden_size}')

## Helper: Get Attention Weights for a Sentence
DistilBERT has **6 transformer layers**, each with **12 attention heads**.
We pass `output_attentions=True` so the model returns attention weight tensors.

In [ ]:
def get_attention(sentence: str):
    """
    Returns:
        tokens     : list of token strings (including [CLS] / [SEP])
        attentions : list of tensors, one per layer, shape (1, num_heads, seq_len, seq_len)
    """
    enc = tokenizer(sentence, return_tensors='pt', truncation=True, max_length=64)
    with torch.no_grad():
        out = model(**enc)
    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    # each element: (1, 12, seq_len, seq_len)
    attentions = [a[0].numpy() for a in out.attentions]  # strip batch dim
    return tokens, attentions


def plot_attention_heatmap(tokens, attn_matrix, title='', ax=None, cbar=False):
    """Plot a single (seq_len x seq_len) attention heatmap."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(attn_matrix, xticklabels=tokens, yticklabels=tokens,
                cmap='Blues', vmin=0, vmax=attn_matrix.max(),
                ax=ax, cbar=cbar, linewidths=0.3, linecolor='lightgrey')
    ax.set_title(title, fontsize=9)
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(tokens, rotation=0, fontsize=7)
    ax.set_xlabel('Key (source token)', fontsize=8)
    ax.set_ylabel('Query (attending token)', fontsize=8)

## 1. Single-Head Attention Heatmap
Each row is a **query** token; the column values show where it attends (how much attention weight it places on each key token).

In [ ]:
sentence = "The cat sat on the mat because it was tired."
tokens, attentions = get_attention(sentence)

print(f'Sentence : "{sentence}"')
print(f'Tokens   : {tokens}')
print(f'Layers   : {len(attentions)}')
print(f'Attention shape per layer: {attentions[0].shape}  (heads, seq_len, seq_len)')

# Plot layer 0, head 0
layer, head = 0, 0
attn = attentions[layer][head]  # (seq_len, seq_len)

fig, ax = plt.subplots(figsize=(9, 7))
plot_attention_heatmap(tokens, attn,
                       title=f'Self-Attention — Layer {layer+1}, Head {head+1}',
                       ax=ax, cbar=True)
plt.tight_layout()
plt.show()

## 2. Average Attention Across All Heads (One Layer)
By averaging over all 12 heads we get a cleaner picture of what the layer broadly attends to.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax_idx, layer_idx in enumerate([0, 2, 5]):   # early, middle, last layer
    avg_attn = attentions[layer_idx].mean(axis=0)  # mean over heads
    plot_attention_heatmap(tokens, avg_attn,
                           title=f'Layer {layer_idx+1} — Mean over 12 heads',
                           ax=axes[ax_idx], cbar=(ax_idx == 2))

plt.suptitle(f'Attention Patterns Across Layers\n"{sentence}"', fontsize=12)
plt.tight_layout()
plt.show()

## 3. Pronoun Resolution — Does Attention Capture "It" → "Cat"?
In *"The cat sat on the mat because it was tired"*, "it" should attend strongly to "cat".
Let's check which tokens the "it" query attends to.

In [ ]:
def plot_token_attention_bar(sentence, target_token, layer_idx=4):
    """Bar chart showing how much `target_token` attends to every other token."""
    tokens, attentions = get_attention(sentence)

    # Find index of target token (first occurrence)
    try:
        tok_idx = next(i for i, t in enumerate(tokens) if target_token.lower() in t.lower())
    except StopIteration:
        print(f'Token "{target_token}" not found in: {tokens}'); return

    # Average over all heads for the chosen layer
    avg_attn = attentions[layer_idx].mean(axis=0)  # (seq_len, seq_len)
    row = avg_attn[tok_idx]  # attention FROM target_token TO all others

    # Plot
    fig, ax = plt.subplots(figsize=(11, 4))
    colors = ['tomato' if t == target_token.lower() else 'steelblue' for t in tokens]
    bars = ax.bar(range(len(tokens)), row, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=40, ha='right', fontsize=10)
    ax.set_ylabel('Attention weight')
    ax.set_title(f'Attention from "{target_token}" to all tokens (Layer {layer_idx+1}, avg over heads)\n"{sentence}"',
                 fontsize=11)
    ax.bar_label(bars, fmt='%.3f', fontsize=8, padding=2)
    plt.tight_layout()
    plt.show()

plot_token_attention_bar(
    "The cat sat on the mat because it was tired.",
    target_token='it', layer_idx=4
)

plot_token_attention_bar(
    "The doctor told the patient that she needed more rest.",
    target_token='she', layer_idx=4
)

## 4. All Heads in One Layer — Side by Side
Each of the 12 heads learns to attend to different relationships.

In [ ]:
short_sentence = "The quick brown fox jumps."
tokens_s, attentions_s = get_attention(short_sentence)

layer_idx = 2  # layer 3 (0-indexed)
n_heads   = attentions_s[layer_idx].shape[0]  # 12

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes_flat = axes.flatten()

for h in range(n_heads):
    attn_h = attentions_s[layer_idx][h]
    sns.heatmap(attn_h,
                xticklabels=tokens_s, yticklabels=tokens_s,
                cmap='Blues', vmin=0, vmax=attn_h.max(),
                ax=axes_flat[h], cbar=False,
                linewidths=0.2, linecolor='lightgrey')
    axes_flat[h].set_title(f'Head {h+1}', fontsize=10)
    axes_flat[h].set_xticklabels(tokens_s, rotation=45, ha='right', fontsize=7)
    axes_flat[h].set_yticklabels(tokens_s, rotation=0, fontsize=7)

plt.suptitle(f'All 12 Attention Heads — Layer {layer_idx+1}\n"{short_sentence}"',
             fontsize=13)
plt.tight_layout()
plt.show()

## 5. Attention Rollout — Global Attention Flow
Because attention is applied layer-by-layer, we can **recursively compose** the attention matrices
to measure how much each input token contributes to any output representation.

**Rollout** (Abnar & Zuidema, 2020): multiply attention matrices layer by layer, adding the identity
at each step to account for residual connections.

In [ ]:
def attention_rollout(attentions):
    """
    Compute attention rollout across all layers.
    attentions: list of (n_heads, seq_len, seq_len) arrays
    Returns: (seq_len, seq_len) rollout matrix
    """
    n_layers = len(attentions)
    seq_len  = attentions[0].shape[-1]

    rollout = np.eye(seq_len)
    for layer_attn in attentions:
        # Average over heads
        avg = layer_attn.mean(axis=0)  # (seq_len, seq_len)
        # Add identity (residual connection)
        avg = avg + np.eye(seq_len)
        # Normalize rows to sum to 1
        avg = avg / avg.sum(axis=-1, keepdims=True)
        rollout = avg @ rollout

    return rollout


sentence2 = "The astronaut landed on the moon and planted a flag there."
tokens2, attentions2 = get_attention(sentence2)

rollout_matrix = attention_rollout(attentions2)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Last-layer average attention
last_avg = attentions2[-1].mean(axis=0)
plot_attention_heatmap(tokens2, last_avg,
                       title='Last Layer — Avg Attention', ax=axes[0], cbar=False)

# Rollout
sns.heatmap(rollout_matrix,
            xticklabels=tokens2, yticklabels=tokens2,
            cmap='YlOrRd', ax=axes[1], cbar=True,
            linewidths=0.3, linecolor='lightgrey')
axes[1].set_title('Attention Rollout (all 6 layers)', fontsize=10)
axes[1].set_xticklabels(tokens2, rotation=45, ha='right', fontsize=7)
axes[1].set_yticklabels(tokens2, rotation=0, fontsize=7)

plt.suptitle(f'"{sentence2}"', fontsize=11)
plt.tight_layout()
plt.show()

## Summary
- Attention weights (`softmax(QK^T/√d_k) V`) show which tokens each query attends to.
- Early layers tend to have local/diagonal attention; later layers develop global, task-specific patterns.
- Individual heads specialize: some attend to the next token, some to semantic relationships.
- Attention rollout propagates attention through all layers to measure global input importance.

> **Next notebook:** Transformer Heads — compare what individual attention heads specialize in.